In [1]:
!pip install polars statsmodels sentence-transformers

In [2]:
import random
import torch
import os

import pandas as pd
import polars as pl
import numpy as np


In [3]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("huggingFaceToken")
login(token=HF_TOKEN)

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/realDataAnalysis/ablation'

Mounted at /content/drive


In [5]:
import sys
sys.path.append(f'{DATA_DIR}')
import compcor.corpus_metrics as corpus_metrics
from compcor.utils import Corpus
from compcor.text_tokenizer_embedder import STTokenizerEmbedder

In [6]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)


In [7]:
# ------------------ Metric Setup (experiment_config.py) ------------------
metrics = [
	corpus_metrics.zero_wasserstein_distance
]

# Helper function for getting metric-dependent data.
def get_metric_dependant_data(metric, corpus: Corpus):
	c = corpus
	return c

# Helper function for getting all metric data.
def get_data_for_compcor_metrics(corpus):
	tokens = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").tokenize_sentences(corpus)
	embeddings = STTokenizerEmbedder(embedding_model_name = "all-MiniLM-L12-v2").embed_sentences(corpus)
	return tokens, embeddings


In [8]:
setA = ['can you tell me how i would normally say thank you as a french person', 'can you translate hi into spanish for me', 'can you translate milk into spanish for me', 'how can i say thank you very much in chinese', 'how can i thank somebody in italian', 'how could i say twin in chinese', 'how do germans say goodnight','how do i ask about the weather in chinese', 'how do i say hotel in finnish', 'how do i say bathroom in italian']
setB = ['how can i say thank you very much in chinese', 'how can i thank somebody in italian', 'how could i say twin in chinese', 'how do they say tacos in mexico', 'how do they say yes in brazil', 'how do vietnameses people say hello', 'how do you say cat in spanish', 'how do you say dog in spanish', 'how do you say fast in spanish', 'how do you say good bye in french', 'how do you say goodbye in spanish', 'how do you say hello in french', 'how do you say hello in japanese', 'how do you say hello in mexico']


In [11]:
def get_distances_from_compare_corpora(setA, setB):
    tokensA, embeddingsA = get_data_for_compcor_metrics(setA)
    tokensB, embeddingsB = get_data_for_compcor_metrics(setB)
    for metric in metrics:
        tempA, tempB = setA, setB
        distances = metric(corpus1=tempA, corpus2=tempB)

    return distances

test_df = get_distances_from_compare_corpora(setA, setB)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

In [14]:
from itertools import combinations
ZERO_SHOT_MODELS = [
"cross-encoder/nli-deberta-v3-small", # low capacity
"typeform/distilbert-base-uncased-mnli", # medium capacity
"valhalla/distilbart-mnli-12-3", # higher capacity
]

# prompt templates
TEMPLATES = {
	"prompt1": "This example is {}.",
	"prompt2": "The writing style of this text is {}.",
	"prompt3": "This text is {}.",
	"prompt4": "This text is written in a {} style.",
	"prompt5": "This text shows {} characteristics."
}


# Make combinations (every possible non-empty model subset)
def all_nonempty_subsets(items):
	return [
		list(combo)
		for r in range(1, len(items) + 1) for combo in combinations(items, r)
	]

model_combinations = all_nonempty_subsets(ZERO_SHOT_MODELS)
prompt_combinations = all_nonempty_subsets(TEMPLATES.keys())

In [22]:
temp

,models,prompts,wasserstein,time
216,"[cross-encoder/nli-deberta-v3-small, typeform/...","[prompt1, prompt2, prompt3, prompt4, prompt5]",0.135053,1.221543


In [27]:
for models in model_combinations:
  for prompts in prompt_combinations:
    print()
    print("NEXT")
    # print(models, prompts)
    temp = test_df[
            test_df['models'].apply(lambda x: x == models) &
            test_df['prompts'].apply(lambda x: x == prompts)
        ]
    print(float(temp['time']))

cross-encoder/nli-deberta-v3-small__prompt1
NEXT
0.07165050506591797
cross-encoder/nli-deberta-v3-small__prompt2
NEXT
0.07082104682922363
cross-encoder/nli-deberta-v3-small__prompt3
NEXT
0.06816744804382324
cross-encoder/nli-deberta-v3-small__prompt4
NEXT
0.07072997093200684
cross-encoder/nli-deberta-v3-small__prompt5
NEXT
0.06914806365966797
cross-encoder/nli-deberta-v3-small__prompt1_prompt2
NEXT
0.13828778266906738
cross-encoder/nli-deberta-v3-small__prompt1_prompt3
NEXT
0.13571381568908691
cross-encoder/nli-deberta-v3-small__prompt1_prompt4
NEXT
0.1383054256439209
cross-encoder/nli-deberta-v3-small__prompt1_prompt5
NEXT
0.13675761222839355
cross-encoder/nli-deberta-v3-small__prompt2_prompt3
NEXT
0.1360478401184082
cross-encoder/nli-deberta-v3-small__prompt2_prompt4
NEXT
0.13862299919128418
cross-encoder/nli-deberta-v3-small__prompt2_prompt5
NEXT
0.1370537281036377
cross-encoder/nli-deberta-v3-small__prompt3_prompt4
NEXT
0.13603496551513672
cross-encoder/nli-deberta-v3-small__prompt

In [13]:
test_df['models'].value_counts(), test_df['prompts'].value_counts()

(models
 [cross-encoder/nli-deberta-v3-small]                                                                          31
 [typeform/distilbert-base-uncased-mnli]                                                                       31
 [valhalla/distilbart-mnli-12-3]                                                                               31
 [cross-encoder/nli-deberta-v3-small, typeform/distilbert-base-uncased-mnli]                                   31
 [cross-encoder/nli-deberta-v3-small, valhalla/distilbart-mnli-12-3]                                           31
 [typeform/distilbert-base-uncased-mnli, valhalla/distilbart-mnli-12-3]                                        31
 [cross-encoder/nli-deberta-v3-small, typeform/distilbert-base-uncased-mnli, valhalla/distilbart-mnli-12-3]    31
 Name: count, dtype: int64,
 prompts
 [prompt1]                                        7
 [prompt2]                                        7
 [prompt3]                                        7
 